In [66]:
import warnings
warnings.filterwarnings('ignore')

# Loading dataset + Preprocessing

In [54]:
# Loading dataset
import pandas as pd
df = pd.read_csv("/content/healthcare-dataset-stroke-data.csv")
df.head(3)

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1


In [55]:
# Dropping missing values
df.dropna(inplace=True)

In [56]:
# One hot encoding categorical columns
df = pd.get_dummies(df)

# Converting boolean columns to float
for col in df.columns:
    if df[col].dtype == bool:
        df[col] = df[col].astype(float)

In [57]:
# Train test split
from sklearn.model_selection import train_test_split
X = df.drop('stroke', axis=1)
y = df['stroke']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [58]:
y_train.value_counts()

,count
stroke,
0,3771
1,156


In [59]:
# Handling imbalance
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [60]:
y_train_resampled.value_counts()

,count
stroke,
0,3771
1,3771


# Fitting model (rf)

In [61]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(class_weight='balanced')
rf.fit(X_train_resampled, y_train_resampled)

RandomForestClassifier(class_weight='balanced')

In [62]:
from sklearn.metrics import classification_report
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      1.00      0.97       929
           1       0.20      0.02      0.03        53

    accuracy                           0.94       982
   macro avg       0.57      0.51      0.50       982
weighted avg       0.91      0.94      0.92       982



# Creating counterfactual explanations

In [ ]:
!pip install dice-ml

In [63]:
import dice_ml

# Dataset
data_dice = dice_ml.Data(
    dataframe=df,
    continuous_features=['age', 'avg_glucose_level', 'bmi'],
    outcome_name='stroke'
)

# Model
rf_dice = dice_ml.Model(
    model=rf,
    backend="sklearn"
)

# Explainer
explainer = dice_ml.Dice(
    data_dice,
    rf_dice,
    method="random"
)

In [64]:
# Select input datapoint
input_datapoint = X_test[0:1]
X_test[0:1]

,id,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,...,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
4336,53802,80.0,0,1,125.32,32.9,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


In [67]:
# Generate counterfactual explanations
cf = explainer.generate_counterfactuals(
    input_datapoint,
    total_CFs=3,
    desired_class="opposite"
)

100%|██████████| 1/1 [00:00<00:00,  2.43it/s]


In [69]:
# Visualize CF explanations
cf.visualize_as_dataframe(show_only_changes=False)

Query instance (original outcome : 0)


,id,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,...,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,stroke
0,53802,80.0,0,1,125.32,32.900002,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0



Diverse Counterfactual set (new outcome: 1)


,id,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,...,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,stroke
0,53802,80.0,0,1,125.32,32.9,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1
1,53802,80.0,0,1,125.32,32.9,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1
2,53802,80.0,0,1,125.32,32.9,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1


In [70]:
# Visualize CF explnations
cf.visualize_as_dataframe(show_only_changes=True)

Query instance (original outcome : 0)


,id,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,...,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,stroke
0,53802,80.0,0,1,125.32,32.900002,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0



Diverse Counterfactual set (new outcome: 1)


,id,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,...,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,stroke
0,-,-,-,-,-,-,-,-,-,-,...,-,-,-,-,-,0.0,-,-,-,1.0
1,-,-,-,-,-,-,-,0.0,-,-,...,-,-,-,-,-,0.0,-,-,-,1.0
2,-,-,-,-,-,-,-,-,-,-,...,-,-,-,-,-,0.0,-,-,-,1.0


In [84]:
# Creating Conditional counterfactuals

# Select input datapoint
i = 205
input_datapoint = X_test[i:i+1]

# Conditions
features_to_vary = ['avg_glucose_level', 'bmi', 'smoking_status_smokes']
permitted_range = {'avg_glucose_level': [40, 300], 'bmi': [15, 45]}

# Generate counterfactual explanations
cf = explainer.generate_counterfactuals(
    input_datapoint,
    total_CFs=3,
    desired_class="opposite",
    features_to_vary=features_to_vary,
    permitted_range=permitted_range
)

# Visualize CF explanations
cf.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [00:00<00:00,  2.14it/s]

Query instance (original outcome : 0)


,id,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,...,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,stroke
0,71777,74.0,1,1,77.160004,26.299999,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0



Diverse Counterfactual set (new outcome: 1)


,id,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,...,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,stroke
0,-,-,-,-,68.87,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,1.0
1,-,-,-,-,69.51,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,1.0
2,-,-,-,-,66.81,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,1.0
